# EpstLoop on a Colab T4

Development stays in VS Code; this notebook only **runs** the GPU stage and sends results back.

1. *Runtime → Change runtime type → T4 GPU*, then run the cells top to bottom.
2. Each run writes into `results/`. The last cell zips them for download. Unzip into the local repo's `results/` folder, and the analysis (`calibration_study.py`) runs locally on CPU.

If a cell fails, copy the full error back to Claude in VS Code.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code (re-run after every push from VS Code)

In [ ]:
import os
REPO = "https://github.com/chiruuu21/epistemic-architecture.git"
if not os.path.exists("/content/epistemic-architecture"):
    !git clone -q {REPO} /content/epistemic-architecture
%cd /content/epistemic-architecture
!git pull -q && git log --oneline -1

## 2. Install and cache models on Google Drive
Caching on Drive means a disconnected session does not re-download the models. Skip the Drive part if you prefer; everything still works without it.

In [ ]:
!pip install -q -r requirements-colab.txt
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

## 3. Sanity check: the offline test suite (should say ALL 70 CHECKS PASSED)

In [ ]:
!python run_all_tests.py

## 4. GPU stage: GPT-2 small (the project's model), 3,000 TriviaQA questions
Takes a few minutes. The final printout includes `accuracy_em` and the loop check, which should be about 1e-3 or smaller.

In [ ]:
!python collect_triviaqa.py --model openai-community/gpt2 --n 3000

## 5. GPU stage: Qwen2.5-3B, the same 3,000 questions
A model that can actually answer trivia, so that ECE has populated bins. It runs in fp16 (about 6 GB). If it stops with *non-finite hidden states*, run the float32 cell below it instead (about 12 GB, still fits on a T4).

In [ ]:
!python collect_triviaqa.py --model Qwen/Qwen2.5-3B --n 3000 --bs 16

In [ ]:
# only if the fp16 run failed with "non-finite hidden states"
# !python collect_triviaqa.py --model Qwen/Qwen2.5-3B --n 3000 --dtype float32 --bs 8

## 6. Analysis (CPU; also runs locally from the same files)

In [ ]:
!python calibration_study.py --tag gpt2
!python calibration_study.py --tag qwen2.5-3b

## 7. Bring the results back to VS Code

In [ ]:
import glob, shutil, zipfile
files = sorted(glob.glob("results/triviaqa_*") + glob.glob("results/calibration_*"))
with zipfile.ZipFile("epstloop_results.zip", "w") as z:
    for f in files:
        z.write(f)
print("\n".join(files))
os.makedirs("/content/drive/MyDrive/EpstLoop_results", exist_ok=True)
shutil.copy("epstloop_results.zip", "/content/drive/MyDrive/EpstLoop_results/")
from google.colab import files as colab_files
colab_files.download("epstloop_results.zip")